# JumpReLU SAE Run Diagnostic Report

Pulls a training run's real metric history from W&B, renders readable charts (matplotlib,
one clear figure per metric — not W&B's own dashboard rendering of histograms/tables), and
produces a table-driven pass/fail verdict plus concrete hyperparameter recommendations.

Runs non-interactively (a fixed `RUN_ID`, no manual pick-and-edit step) so this can be
executed headlessly via `uv run jupyter nbconvert --to notebook --execute`, not just in an
interactive kernel. Figures are saved as PNGs under `OUT_DIR` so they're inspectable even from
an executed-but-not-manually-viewed notebook.

In [ ]:
# 1. Setup & Auth
# Requires WANDB_API_KEY in the environment or a prior `wandb login` — never type the key
# into a cell.
import sys
from pathlib import Path

if str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

import pandas as pd
from IPython.display import Image, display

from ptm_sae.analysis import fetch_run, plot_checks, pull_histogram, pull_scalars, run_diagnostics

PROJECT = "ptm-sae"

In [ ]:
# 2. Locate the Run
# Lists every run in the project so a different bad run can be swapped in later just by
# changing RUN_ID below — no interactive pick-and-edit step needed to run this end to end.
import wandb

for run in wandb.Api().runs(PROJECT):
    print(run.id, run.name, run.created_at, run.state, run.tags)

RUN_ID = "6titmmxz"  # jumprelu_w4096, the run under diagnosis

In [ ]:
# 3. Pull Metric History
run = fetch_run(PROJECT, RUN_ID)
SAE_TYPE = run.config["sae_type"]

scalar_keys = [
    "train/threshold_std", "train/threshold_mean", "train/l0",
    "train/decoder_pre_norm_mean",
    "val/explained_variance", "val/cosine_sim_mean", "val/cosine_sim_p10",
    "val/mse", "val/dead_latent_fraction", "val/alive_latent_jaccard",
]
scalars = pull_scalars(run, scalar_keys)
histogram = pull_histogram(run, "val/feature_density_histogram")

print(f"sae_type={SAE_TYPE}  rows={len(scalars)}  final_step={scalars.index.max()}")
scalars.tail()

In [ ]:
# 4. Diagnostic Plots
OUT_DIR = Path("jumprelu_diagnostic_report_figures")
figure_paths = plot_checks(
    scalars, SAE_TYPE, OUT_DIR, histogram=histogram, l0_warmup_steps=run.config.get("l0_warmup_steps")
)
for path in figure_paths:
    display(Image(filename=str(path)))

In [ ]:
# 5. Verdict
result = run_diagnostics(scalars, SAE_TYPE, run.config)
pd.set_option("display.max_colwidth", None)
result.verdict

In [ ]:
# 6. Recommended Config Changes
for observed, diagnosis, recommendation in result.recommendations:
    print(f"- {observed}")
    if diagnosis:
        print(f"  diagnosis:   {diagnosis}")
    if recommendation:
        print(f"  recommend:   {recommendation}")
    print()